### Build a Question Answering application over a Graph Database

In [16]:
NEO4J_URI="neo4j+s://93d4d6a3.databases.neo4j.io"
NEO4J_USERNAME="93d4d6a3"
NEO4J_PASSWORD="VqggSGimS2U5mgEcgBPlnuSh_2_I55qzU5JwXZEov1A"
NEO4J_DATABASE="93d4d6a3"

In [17]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "neo4j+ssc://93d4d6a3.databases.neo4j.io",  # 👈 CHANGE HERE
    auth=("93d4d6a3", "VqggSGimS2U5mgEcgBPlnuSh_2_I55qzU5JwXZEov1A")
)

driver.verify_connectivity()
print("Connected ✅")

Connected ✅


In [18]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url="neo4j+ssc://93d4d6a3.databases.neo4j.io",
    username=NEO4J_USERNAME,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

In [19]:
import os
os.environ["NEO4J_URI"] = NEO4J_URI
os.environ["NEO4J_USERNAME"] = NEO4J_USERNAME
os.environ["NEO4J_PASSWORD"] = NEO4J_PASSWORD
os.environ["NEO4J_DATABASE"] = NEO4J_DATABASE

In [20]:
graph

In [21]:
## Dataset Moview 
moview_query="""
LOAD CSV WITH HEADERS FROM
'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row

MERGE(m:Movie{id:row.movieId})
SET m.released = date(row.released),
    m.title = row.title,
    m.imdbRating = toFloat(row.imdbRating)
FOREACH (director in split(row.director, '|') | 
    MERGE (p:Person {name:trim(director)})
    MERGE (p)-[:DIRECTED]->(m))
FOREACH (actor in split(row.actors, '|') | 
    MERGE (p:Person {name:trim(actor)})
    MERGE (p)-[:ACTED_IN]->(m))
FOREACH (genre in split(row.genres, '|') | 
    MERGE (g:Genre {name:trim(genre)})
    MERGE (m)-[:IN_GENRE]->(g))


"""

In [22]:
moview_query

"\nLOAD CSV WITH HEADERS FROM\n'https://raw.githubusercontent.com/tomasonjo/blog-datasets/main/movies/movies_small.csv' as row\n\nMERGE(m:Movie{id:row.movieId})\nSET m.released = date(row.released),\n    m.title = row.title,\n    m.imdbRating = toFloat(row.imdbRating)\nFOREACH (director in split(row.director, '|') | \n    MERGE (p:Person {name:trim(director)})\n    MERGE (p)-[:DIRECTED]->(m))\nFOREACH (actor in split(row.actors, '|') | \n    MERGE (p:Person {name:trim(actor)})\n    MERGE (p)-[:ACTED_IN]->(m))\nFOREACH (genre in split(row.genres, '|') | \n    MERGE (g:Genre {name:trim(genre)})\n    MERGE (m)-[:IN_GENRE]->(g))\n\n\n"

In [23]:
graph.query(moview_query)

[]

In [24]:
graph.refresh_schema()
print(graph.schema)

Node properties:
Movie {id: STRING, released: DATE, title: STRING, imdbRating: FLOAT}
Person {name: STRING}
Genre {name: STRING}
Relationship properties:

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)


In [40]:
from dotenv import load_dotenv
import os

load_dotenv()

print(os.getenv("GROQ_API_KEY"))  # must print key

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


In [53]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model= "llama-3.1-8b-instant"
)

In [54]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    verbose=True,
    allow_dangerous_requests=True
)

In [55]:
response = chain.invoke({
    "query": "Who was the director of the movie Casino?"
})

print(response)



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: "Casino"})-[:DIRECTED]->(p:Person) RETURN p.name;
Full Context:
[]

> Finished chain.
{'query': 'Who was the director of the movie Casino?', 'result': "I don't know the answer."}


In [56]:
response=chain.invoke({"query":"Who were the actors of the movie Casino"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie {title: "Casino"})-[:ACTED_IN]->(a:Person) RETURN a.name
Full Context:
[]

> Finished chain.


{'query': 'Who were the actors of the movie Casino',
 'result': "I don't know the answer."}

In [57]:
response=chain.invoke({"query":"How many artists are there?"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person) RETURN COUNT(p) AS Count
Full Context:
[{'Count': 1239}]

> Finished chain.


{'query': 'How many artists are there?', 'result': "I don't know the answer."}

In [58]:
response=chain.invoke({"query":"How many movies has Tom Hanks acted in"})
response



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (m:Movie)-[r:ACTED_IN]->(p:Person) WHERE p.name = "Tom Hanks" RETURN COUNT(m)
Full Context:
[{'COUNT(m)': 0}]

> Finished chain.


{'query': 'How many movies has Tom Hanks acted in',
 'result': "I don't know the answer."}